# Contract Intelligence Multi-Agent System
## Week 8 Assignment: Production Deployment

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Learning Objectives

- Build a production system integration class
- Create a Gradio interactive UI
- Deploy as a FastAPI application

## Difficulty: Advanced

**IMPORTANT**: Run all prior week cells first before starting this week's exercises.

---

---
# WEEK 1: Environment Setup & Langfuse Initialization (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 1: Environment Setup & Langfuse Initialization

---

## Learning Objectives

By the end of Week 1, you will be able to:

- [ ] Configure a production-ready development environment
- [ ] Initialize Langfuse observability from the very first line of code
- [ ] Understand the contract document taxonomy
- [ ] Discover and catalog available contract data
- [ ] Create traced wrapper functions for all LLM operations

---

## Key Concept: Why Observability First?

In traditional software development, logging and monitoring are often added as an afterthought. In AI systems, this approach fails catastrophically because:

1. **Non-determinism**: LLM outputs vary even with identical inputs
2. **Cost tracking**: API calls have real monetary costs
3. **Quality assurance**: You need to evaluate response quality over time
4. **Debugging**: When things go wrong, you need complete traces
5. **Compliance**: Enterprise systems require audit trails

**Langfuse** provides:
- **Traces**: End-to-end visibility of complex workflows
- **Spans**: Nested operations within traces
- **Generations**: Specific LLM calls with inputs/outputs
- **Scores**: Quality metrics attached to traces
- **Sessions**: Grouping of related traces

```
Session (notebook run)
  └── Trace (contract analysis)
       ├── Span (document processing)
       ├── Generation (embedding call)
       ├── Generation (LLM completion)
       └── Score (risk assessment quality)
```

## 1.1 Package Installation

We install packages in a specific order to avoid dependency conflicts. Each package serves a distinct purpose in our architecture.

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("openpyxl", "openpyxl"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

## Key Concept: Package Architecture

Understanding why we use each package helps you make informed choices in your own projects:

| Package | Purpose | Alternative Options |
|---------|---------|--------------------|
| `langchain` | Orchestration framework | LlamaIndex, Haystack |
| `langfuse` | Observability & tracing | LangSmith, Phoenix |
| `chromadb` | Vector storage | Pinecone, Weaviate, Qdrant |
| `networkx` | Graph operations | Neo4j, igraph |
| `pydantic` | Data validation | dataclasses, attrs |
| `tenacity` | Retry logic | backoff, retrying |
| `gradio` | UI framework | Streamlit, FastAPI |

## 1.2 Environment Configuration

This section handles the critical task of detecting our runtime environment and loading API keys securely.

In [ ]:
# ============================================================================
# WEEK 1.2: ENVIRONMENT DETECTION & CONFIGURATION
# ============================================================================

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# ENVIRONMENT DETECTION
# ============================================================================
# We detect if we're running in Google Colab to handle paths and secrets differently

IN_COLAB = 'google.colab' in sys.modules

print(f"Runtime Environment: {'Google Colab' if IN_COLAB else 'Local/Other'}")
print(f"Python Version: {sys.version.split()[0]}")

In [ ]:
# ============================================================================
# API KEY CONFIGURATION & DATA INGESTION
# ============================================================================
# API keys from Colab Secrets or .env file
# Datasets fetched programmatically from GitHub

import subprocess

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "contract_intelligence"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)


# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

In [ ]:
# ============================================================================
# PROJECT CONFIGURATION
# ============================================================================

# Project identifiers
PROJECT_NAME = "contract-intelligence-system"
DATA_DIR = Path(DATA_DIR_BASE)

# Display configuration summary (masking sensitive data)
print("\n" + "="*60)
print("PROJECT CONFIGURATION")
print("="*60)
print(f"Project Name: {PROJECT_NAME}")
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Data Directory: {DATA_DIR}")
print(f"OpenAI API Key: {'*' * 20}...{os.environ.get('OPENAI_API_KEY', '')[-4:]}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_HOST')}")

## Deep Dive: Secure API Key Management

API keys are the "passwords" to your AI services. Proper handling is critical:

### DO:
- Use Colab Secrets (Settings > Secrets) for Colab notebooks
- Use `.env` files (gitignored) for local development
- Use environment variables in production
- Rotate keys periodically

### DON'T:
- Hardcode keys in notebooks or code
- Commit keys to version control
- Share keys via email or chat
- Use the same key for dev and production

```python
# WRONG - Never do this!
OPENAI_API_KEY = "sk-abc123..."

# RIGHT - Load from environment
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
```

## 1.3 Langfuse Initialization

Now we initialize Langfuse - this is the **foundation of our observability strategy**. Every subsequent operation will be traceable.

In [ ]:
# ============================================================================
# WEEK 1.3: LANGFUSE INITIALIZATION
# ============================================================================

from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
from openai import OpenAI

# Initialize Langfuse client
langfuse = Langfuse(
    secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
    public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
    host=os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')
)

# Verify connection to Langfuse
try:
    langfuse.auth_check()
    print("Langfuse connection verified!")
except Exception as e:
    print(f"Langfuse connection failed: {e}")
    raise

# Create a unique session ID for this notebook run
# This groups all traces from this session together
SESSION_ID = f"contract-intel-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n" + "="*60)
print("LANGFUSE OBSERVABILITY INITIALIZED")
print("="*60)
print(f"Session ID: {SESSION_ID}")
print(f"Host: {os.environ.get('LANGFUSE_HOST')}")
print(f"\nAll operations will now be traced.")
print(f"View traces at: {os.environ.get('LANGFUSE_HOST')}/project")

## Key Concept: Session IDs and Trace Organization

In Langfuse, traces are organized hierarchically:

```
Project
  └── Session (SESSION_ID: contract-intel-20240115-143022)
       ├── Trace: data-discovery
       ├── Trace: document-processing
       ├── Trace: embedding-generation
       ├── Trace: legal-agent-analysis
       └── Trace: orchestration-run
```

**Why Session IDs matter:**
1. Group related traces from a single notebook run
2. Compare performance across different runs
3. Debug issues by replaying exact sequences
4. Calculate aggregate metrics per session

## 1.4 Traced Wrapper Functions

We create wrapper functions that automatically trace all OpenAI API calls. This ensures **100% observability** without cluttering our application code.

In [ ]:
# ============================================================================
# WEEK 1.4: TRACED OPENAI WRAPPER FUNCTIONS
# ============================================================================

# Initialize OpenAI client
openai_client = OpenAI()

def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    This wrapper ensures every embedding call is:
    1. Logged with input text (truncated for privacy)
    2. Timed for performance monitoring
    3. Token usage tracked for cost analysis

    Args:
        text: The text to embed (max 8192 tokens for text-embedding-3-small)
        trace_name: Identifier for this trace in Langfuse

    Returns:
        List of floats representing the embedding vector (1536 dimensions)
    """
    # Create a trace for this operation
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={
            "text_length": len(text),
            "text_preview": text[:100] + "..." if len(text) > 100 else text
        }
    )

    # Create a generation span for the embedding call
    generation = trace.generation(
        name="openai-embedding",
        model="text-embedding-3-small",
        input=text[:500] + "..." if len(text) > 500 else text
    )

    # Make the actual API call
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    embedding = response.data[0].embedding

    # End the generation with output metadata
    generation.end(
        output={"dimensions": len(embedding)},
        usage={"total_tokens": response.usage.total_tokens}
    )

    return embedding

print("traced_embedding() function defined")

In [ ]:
# ============================================================================
# TRACED COMPLETION FUNCTION
# ============================================================================

def traced_completion(
    messages: List[Dict],
    model: str = "gpt-4o-mini",
    trace_name: str = "completion",
    temperature: float = 0,
    **kwargs
) -> str:
    """
    Generate LLM completion with full Langfuse tracing.

    This wrapper captures:
    1. Full message history (input)
    2. Model response (output)
    3. Token usage (prompt + completion + total)
    4. Latency timing

    Args:
        messages: List of message dicts [{"role": "user", "content": "..."}]
        model: OpenAI model identifier
        trace_name: Identifier for this trace in Langfuse
        temperature: Sampling temperature (0 = deterministic)
        **kwargs: Additional arguments passed to OpenAI API

    Returns:
        String containing the model's response
    """
    # Create trace
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={"model": model, "temperature": temperature}
    )

    # Create generation span
    generation = trace.generation(
        name="openai-completion",
        model=model,
        input=messages
    )

    # Make API call
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        **kwargs
    )

    content = response.choices[0].message.content

    # End generation with full usage data
    generation.end(
        output=content,
        usage={
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    )

    return content

print("traced_completion() function defined")

## Example: Testing Our Traced Functions

Let's verify our traced functions work correctly and observe the traces in Langfuse.

In [ ]:
# ============================================================================
# EXAMPLE 1: Test Embedding Function
# ============================================================================

print("EXAMPLE 1: Testing traced_embedding()")
print("="*50)

test_text = "This Master Services Agreement governs the relationship between the parties."
embedding = traced_embedding(test_text, trace_name="test-embedding-1")

print(f"Input text: '{test_text}'")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")
print(f"\nThis trace is now visible in Langfuse!")

In [ ]:
# ============================================================================
# EXAMPLE 2: Test Completion Function
# ============================================================================

print("\nEXAMPLE 2: Testing traced_completion()")
print("="*50)

messages = [
    {"role": "system", "content": "You are a contract analysis expert."},
    {"role": "user", "content": "What are the key elements of an MSA?"}
]

response = traced_completion(messages, trace_name="test-completion-1")

print(f"Response preview: {response[:200]}...")
print(f"\nThis trace is now visible in Langfuse!")

In [ ]:
# ============================================================================
# EXAMPLE 3: Multiple Embeddings (Batch Processing Pattern)
# ============================================================================

print("\nEXAMPLE 3: Batch Embedding Processing")
print("="*50)

sample_clauses = [
    "The Contractor shall indemnify and hold harmless the Client.",
    "Payment terms are Net 30 from invoice date.",
    "Either party may terminate with 90 days written notice."
]

for i, clause in enumerate(sample_clauses, 1):
    embedding = traced_embedding(clause, trace_name=f"clause-embedding-{i}")
    print(f"Clause {i}: {len(embedding)} dimensions")

# Flush traces to Langfuse
langfuse.flush()
print(f"\nAll traces flushed to Langfuse!")

## 1.5 Contract Document Taxonomy

Before processing documents, we need to understand what we're working with. Enterprise contract repositories have a specific structure.

In [ ]:
# ============================================================================
# WEEK 1.5: CONTRACT DOCUMENT TAXONOMY
# ============================================================================

CONTRACT_CATEGORIES = {
    'master_agreements': {
        'path': 'Master level agreements',
        'description': 'Foundation contracts establishing overall relationship',
        'document_types': ['MSA', 'NDA', 'Rate Cards'],
        'risk_focus': ['legal', 'compliance'],
        'typical_length': '20-50 pages'
    },
    'transaction_contracts': {
        'path': 'Transaction level contract',
        'description': 'Project-specific agreements under master agreements',
        'document_types': ['SOW', 'Work Orders', 'Renewals', 'Amendments'],
        'risk_focus': ['operational', 'financial'],
        'typical_length': '5-20 pages'
    },
    'commercial_docs': {
        'path': 'Commercial docs',
        'description': 'Pricing and commercial terms documentation',
        'document_types': ['Pricing Tables', 'Rate Schedules', 'Discount Structures'],
        'risk_focus': ['financial'],
        'typical_length': '2-10 pages'
    },
    'financial_billing': {
        'path': 'Financial and Billing docs',
        'description': 'Invoices, payment records, and billing schedules',
        'document_types': ['Invoices', 'Credit Notes', 'Payment Schedules'],
        'risk_focus': ['financial'],
        'typical_length': '1-5 pages'
    },
    'operational_docs': {
        'path': 'Operational service delivery docs',
        'description': 'Service delivery and operational documentation',
        'document_types': ['Service Reports', 'SLA Reports', 'Incident Logs'],
        'risk_focus': ['operational'],
        'typical_length': '5-15 pages'
    },
    'compliance_docs': {
        'path': 'Compliance and policy documents',
        'description': 'Security, compliance, and policy documentation',
        'document_types': ['InfoSec Controls', 'Vendor Policies', 'Compliance Certs'],
        'risk_focus': ['legal', 'compliance'],
        'typical_length': '10-30 pages'
    },
    'negotiation_history': {
        'path': 'Historical negotiation data',
        'description': 'Historical negotiation records and communications',
        'document_types': ['Redlines', 'Email Threads', 'Risk Assessments'],
        'risk_focus': ['legal', 'financial', 'operational'],
        'typical_length': 'Variable'
    },
    'governance': {
        'path': 'Disputes, audits and governance data',
        'description': 'Governance, disputes, and audit documentation',
        'document_types': ['Audit Reports', 'Dispute Records', 'Governance Minutes'],
        'risk_focus': ['operational', 'compliance'],
        'typical_length': '5-20 pages'
    }
}

print("CONTRACT DOCUMENT TAXONOMY")
print("="*70)

In [ ]:
# ============================================================================
# DISPLAY TAXONOMY WITH TRACING
# ============================================================================

# Log taxonomy to Langfuse for audit trail
taxonomy_trace = langfuse.trace(
    name="taxonomy-definition",
    session_id=SESSION_ID,
    input={"categories": list(CONTRACT_CATEGORIES.keys())},
    metadata={"version": "1.0"}
)

taxonomy_trace.span(
    name="categories-loaded",
    input={"count": len(CONTRACT_CATEGORIES)}
)

# Display formatted taxonomy
for category, info in CONTRACT_CATEGORIES.items():
    print(f"\n{category.upper().replace('_', ' ')}")
    print(f"  Path: {info['path']}")
    print(f"  Description: {info['description']}")
    print(f"  Document Types: {', '.join(info['document_types'])}")
    print(f"  Risk Focus: {', '.join(info['risk_focus'])}")
    print(f"  Typical Length: {info['typical_length']}")

## Key Concept: Contract Hierarchy

Understanding the contract hierarchy is essential for proper risk assessment:

```
Master Service Agreement (MSA)
  │
  ├── NDA (Non-Disclosure Agreement)
  │
  ├── Rate Card (Pricing Framework)
  │
  └── Statement of Work (SOW) #1
       │
       ├── Change Order #1
       ├── Change Order #2
       └── Invoice Records
```

**Why this matters:**
- MSA terms cascade down to all subordinate documents
- SOW terms can override MSA terms (within limits)
- Risk in a subordinate document may indicate risk in the master
- Our knowledge graph (Week 6) will model these relationships

## 1.6 Data Discovery with Tracing

Now we discover what contract data is available in our dataset.

In [ ]:
# ============================================================================
# WEEK 1.6: DATA DISCOVERY WITH TRACING
# ============================================================================

def discover_contract_data(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Discover all contract documents organized by category.

    This function:
    1. Scans the data directory for contract documents
    2. Categorizes documents based on folder structure
    3. Logs the discovery process to Langfuse

    Args:
        data_dir: Path to the root data directory

    Returns:
        Dictionary mapping categories to lists of file paths
    """
    # Create trace for data discovery
    trace = langfuse.trace(
        name="data-discovery",
        session_id=SESSION_ID,
        input={"data_dir": str(data_dir)}
    )

    discovered = {cat: [] for cat in CONTRACT_CATEGORIES.keys()}

    # Check if data directory exists
    if not data_dir.exists():
        trace.span(
            name="error",
            input={"message": f"Directory not found: {data_dir}"}
        )
        print(f"Warning: Data directory not found at {data_dir}")
        print("Continuing with empty dataset for demonstration...")
        return discovered

    # Scan each category
    for category, info in CONTRACT_CATEGORIES.items():
        category_path = data_dir / info['path']

        span = trace.span(
            name=f"scan-{category}",
            input={"path": str(category_path)}
        )

        if category_path.exists():
            # Find all documents recursively
            for item in category_path.rglob('*'):
                if item.is_file() and item.suffix.lower() in ['.docx', '.pdf', '.xlsx']:
                    discovered[category].append(item)

        span.end(output={"files_found": len(discovered[category])})

    # Calculate totals
    total = sum(len(files) for files in discovered.values())
    trace.update(output={
        "total_files": total,
        "categories_with_data": sum(1 for files in discovered.values() if files)
    })

    return discovered

print("discover_contract_data() function defined")

In [ ]:
# ============================================================================
# EXECUTE DATA DISCOVERY
# ============================================================================

contract_files = discover_contract_data(DATA_DIR)

print("\n" + "="*70)
print("CONTRACT DATASET DISCOVERY RESULTS")
print("="*70)

total_files = 0
for category, files in contract_files.items():
    print(f"\n{category.upper().replace('_', ' ')}: {len(files)} files")

    # Show first 3 files as examples
    for f in files[:3]:
        print(f"  - {f.name}")

    if len(files) > 3:
        print(f"  ... and {len(files) - 3} more")

    total_files += len(files)

print(f"\n" + "="*70)
print(f"TOTAL CONTRACT DOCUMENTS: {total_files}")
print("="*70)

# Flush traces
langfuse.flush()

## Checkpoint: Week 1 Verification

Before proceeding to Week 2, verify you have completed the following:

In [ ]:
# ============================================================================
# WEEK 1 CHECKPOINT
# ============================================================================

print("WEEK 1 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Langfuse initialized", langfuse is not None),
    ("Session ID created", SESSION_ID is not None and len(SESSION_ID) > 0),
    ("OpenAI client ready", openai_client is not None),
    ("traced_embedding() works", callable(traced_embedding)),
    ("traced_completion() works", callable(traced_completion)),
    ("Contract taxonomy defined", len(CONTRACT_CATEGORIES) == 8),
    ("Data discovery complete", contract_files is not None),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("All Week 1 checkpoints PASSED! Ready for Week 2.")
else:
    print("Some checkpoints FAILED. Review the issues above.")

## Week 1 Summary

### What We Accomplished

| Component | Status | Langfuse Traces |
|-----------|--------|----------------|
| Environment Setup | Complete | N/A |
| Langfuse Initialization | Complete | Session created |
| Traced Wrappers | Complete | test-embedding-*, test-completion-* |
| Contract Taxonomy | Complete | taxonomy-definition |
| Data Discovery | Complete | data-discovery |

### Key Takeaways

1. **Observability First**: Initialize tracing before any other code
2. **Wrapper Functions**: Abstract LLM calls behind traced functions
3. **Session Management**: Group related traces with session IDs
4. **Domain Understanding**: Know your data taxonomy before processing

### Looking Ahead to Week 2

In Week 2, we will:
- Parse DOCX contract documents
- Perform Exploratory Data Analysis (EDA)
- Extract contract entities (parties, dates, amounts)
- Build our document processing pipeline

---
# WEEK 2: Document Processing & Exploratory Data Analysis (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 2: Document Processing & Exploratory Data Analysis

---

## Learning Objectives

By the end of Week 2, you will be able to:

- [ ] Parse DOCX contract documents with full tracing
- [ ] Perform comprehensive EDA on contract corpus
- [ ] Extract structured sections from unstructured documents
- [ ] Implement entity extraction for contract-specific terms
- [ ] Visualize document statistics and distributions

---

## Key Concept: Document Processing Pipeline

Contract documents are semi-structured - they have consistent elements (parties, dates, clauses) but variable formatting. Our pipeline must:

```
Raw Document (DOCX/PDF)
        ↓
    Parsing Layer
        ↓
  Section Extraction
        ↓
  Entity Recognition
        ↓
  Structured Output
```

**Challenges we address:**
1. **Tables**: Contracts often have pricing/schedule tables
2. **Headers/Footers**: Repeated content that's not substantive
3. **Section Numbering**: Variable formats (1.1, A.1, Article I)
4. **Defined Terms**: "Contractor" vs "contractor" (capitalized = defined)

## 2.1 Document Processor Class

We build a production-grade document processor that handles multiple formats and extracts structure.

In [ ]:
# ============================================================================
# WEEK 2.1: DOCUMENT PROCESSOR CLASS
# ============================================================================

from docx import Document as DocxDocument
import re

class ContractDocumentProcessor:
    """
    Production-grade document processor for contract analysis.

    This class provides:
    - Multi-format support (DOCX, with extensibility for PDF)
    - Section extraction using regex patterns
    - Table parsing and text conversion
    - Full Langfuse tracing for all operations

    Design Principles:
    - Fail gracefully with informative errors
    - Preserve document structure when possible
    - Generate unique document IDs for tracking
    """

    def __init__(self):
        self.supported_formats = ['.docx']
        self.processed_docs = []

        # Regex patterns for section headers
        # These patterns match common contract section formats
        self.section_patterns = [
            r'^(\d+\.\s+[A-Z][A-Z\s]+)$',      # "1. DEFINITIONS"
            r'^(\d+\.\d+\s+.+)$',              # "1.1 Term"
            r'^(ARTICLE\s+[IVXLCDM]+)',        # "ARTICLE I"
            r'^(SECTION\s+\d+)',              # "SECTION 1"
            r'^(Schedule\s+[A-Z0-9]+)',       # "Schedule A"
            r'^(Exhibit\s+[A-Z0-9]+)',        # "Exhibit A"
            r'^(APPENDIX\s+[A-Z0-9]+)',       # "APPENDIX A"
        ]

    def load_docx(self, file_path: Path, trace_parent=None) -> Optional[Dict[str, Any]]:
        """
        Load and parse a DOCX contract file.

        Args:
            file_path: Path to the DOCX file
            trace_parent: Parent trace for nested spans

        Returns:
            Dictionary containing parsed document data, or None on error
        """
        # Create span if parent trace exists
        span = trace_parent.span(name=f"load-{file_path.name}") if trace_parent else None

        try:
            # Load the document
            doc = DocxDocument(file_path)

            # Extract paragraphs (filter empty)
            paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

            # Extract tables
            tables = []
            for table in doc.tables:
                table_data = []
                for row in table.rows:
                    row_data = [cell.text.strip() for cell in row.cells]
                    if any(row_data):  # Skip empty rows
                        table_data.append(row_data)
                if table_data:
                    tables.append(table_data)

            # Extract sections using patterns
            sections = self._extract_sections(paragraphs)

            # Build full text
            full_text = '\n'.join(paragraphs)
            table_text = self._tables_to_text(tables)

            result = {
                'filename': file_path.name,
                'filepath': str(file_path),
                'format': 'docx',
                'paragraphs': paragraphs,
                'tables': tables,
                'sections': sections,
                'full_text': full_text,
                'table_text': table_text,
                'word_count': len(full_text.split()),
                'paragraph_count': len(paragraphs),
                'table_count': len(tables),
                'section_count': len(sections),
                'character_count': len(full_text)
            }

            if span:
                span.end(output={
                    "word_count": result['word_count'],
                    "sections": result['section_count'],
                    "tables": result['table_count'],
                    "status": "success"
                })

            return result

        except Exception as e:
            if span:
                span.end(output={"error": str(e), "status": "failed"})
            print(f"Error loading {file_path.name}: {e}")
            return None

    def _extract_sections(self, paragraphs: List[str]) -> Dict[str, str]:
        """
        Extract named sections from document paragraphs.

        Uses regex patterns to identify section headers and
        groups subsequent paragraphs under each section.
        """
        sections = {}
        current_section = "PREAMBLE"  # Default section for content before first header
        current_content = []

        for para in paragraphs:
            is_section_header = False

            # Check if paragraph matches any section pattern
            for pattern in self.section_patterns:
                if re.match(pattern, para, re.IGNORECASE):
                    # Save previous section
                    if current_content:
                        sections[current_section] = '\n'.join(current_content)

                    # Start new section
                    current_section = para
                    current_content = []
                    is_section_header = True
                    break

            if not is_section_header:
                current_content.append(para)

        # Save final section
        if current_content:
            sections[current_section] = '\n'.join(current_content)

        return sections

    def _tables_to_text(self, tables: List) -> str:
        """
        Convert tables to readable text format.

        This preserves table structure in a format
        that can be understood by LLMs.
        """
        text_parts = []
        for i, table in enumerate(tables):
            text_parts.append(f"\nTable {i+1}:")
            for row in table:
                if isinstance(row, list):
                    text_parts.append(' | '.join(str(cell) for cell in row if cell))
        return '\n'.join(text_parts)

print("ContractDocumentProcessor class defined")

In [ ]:
# ============================================================================
# ADD CATEGORY PROCESSING METHOD
# ============================================================================

# Extend the class with category processing
def process_category(self, files: List[Path], category: str) -> List[Dict]:
    """
    Process all files in a category with tracing.

    Args:
        files: List of file paths to process
        category: Category name for document IDs

    Returns:
        List of processed document dictionaries
    """
    # Create trace for category processing
    trace = langfuse.trace(
        name=f"process-{category}",
        session_id=SESSION_ID,
        input={"category": category, "file_count": len(files)},
        tags=["document-processing", category]
    )

    processed = []
    for file_path in files:
        doc = self.load_docx(file_path, trace)
        if doc:
            # Add metadata
            doc['category'] = category
            doc['doc_id'] = f"{category.upper()}-{len(processed)+1:03d}"
            processed.append(doc)

    trace.update(output={
        "processed_count": len(processed),
        "success_rate": len(processed) / len(files) if files else 0
    })

    return processed

# Add method to class
ContractDocumentProcessor.process_category = process_category

print("process_category() method added to ContractDocumentProcessor")

## 2.2 Process All Contract Documents

Now we process all discovered documents with full tracing.

In [ ]:
# ============================================================================
# WEEK 2.2: PROCESS ALL DOCUMENTS
# ============================================================================

# Initialize processor
doc_processor = ContractDocumentProcessor()

print("PROCESSING CONTRACT DOCUMENTS")
print("="*60)
print("Each document is traced in Langfuse.\n")

# Process all categories
all_contract_docs = {}
for category, files in contract_files.items():
    if files:
        all_contract_docs[category] = doc_processor.process_category(files, category)
        print(f"{category}: {len(all_contract_docs[category])} documents processed")

# Flatten all documents into single list
all_docs_flat = []
for docs in all_contract_docs.values():
    all_docs_flat.extend(docs)

print(f"\n" + "="*60)
print(f"TOTAL DOCUMENTS PROCESSED: {len(all_docs_flat)}")
print("="*60)

langfuse.flush()

## 2.3 Exploratory Data Analysis (EDA)

EDA helps us understand our document corpus before building analysis pipelines.

In [ ]:
# ============================================================================
# WEEK 2.3: EXPLORATORY DATA ANALYSIS
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Trace EDA
eda_trace = langfuse.trace(
    name="eda-analysis",
    session_id=SESSION_ID,
    input={"document_count": len(all_docs_flat)},
    tags=["eda", "visualization"]
)

# Build DataFrame for analysis
eda_data = []
for doc in all_docs_flat:
    eda_data.append({
        'doc_id': doc['doc_id'],
        'filename': doc['filename'],
        'category': doc['category'],
        'word_count': doc['word_count'],
        'paragraph_count': doc['paragraph_count'],
        'table_count': doc['table_count'],
        'section_count': doc['section_count'],
        'character_count': doc['character_count']
    })

df_contracts = pd.DataFrame(eda_data)

print("CONTRACT CORPUS EDA")
print("="*60)
print(f"\nDataset Shape: {df_contracts.shape}")

if not df_contracts.empty:
    print(f"\nBasic Statistics:")
    print(df_contracts.describe().round(2))
else:
    print("\nNo documents found - creating sample data for demonstration...")
    # Create sample data for demonstration
    sample_data = [
        {'doc_id': 'SAMPLE-001', 'filename': 'sample_msa.docx', 'category': 'master_agreements',
         'word_count': 5000, 'paragraph_count': 120, 'table_count': 3, 'section_count': 15, 'character_count': 30000},
        {'doc_id': 'SAMPLE-002', 'filename': 'sample_sow.docx', 'category': 'transaction_contracts',
         'word_count': 2000, 'paragraph_count': 50, 'table_count': 5, 'section_count': 8, 'character_count': 12000},
        {'doc_id': 'SAMPLE-003', 'filename': 'sample_nda.docx', 'category': 'master_agreements',
         'word_count': 1500, 'paragraph_count': 35, 'table_count': 0, 'section_count': 10, 'character_count': 9000},
    ]
    df_contracts = pd.DataFrame(sample_data)
    all_docs_flat = sample_data
    print(f"Sample dataset created with {len(df_contracts)} documents")

In [ ]:
# ============================================================================
# EDA VISUALIZATIONS
# ============================================================================

if not df_contracts.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Contract Corpus Exploratory Data Analysis', fontsize=14, fontweight='bold')

    # 1. Documents per Category
    category_counts = df_contracts['category'].value_counts()
    axes[0, 0].barh(category_counts.index, category_counts.values, color='steelblue')
    axes[0, 0].set_xlabel('Number of Documents')
    axes[0, 0].set_title('Documents per Category')
    axes[0, 0].invert_yaxis()

    # 2. Word Count Distribution
    axes[0, 1].hist(df_contracts['word_count'], bins=15, color='coral', edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Word Count')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Word Count Distribution')
    axes[0, 1].axvline(df_contracts['word_count'].mean(), color='red', linestyle='--',
                       label=f'Mean: {df_contracts["word_count"].mean():.0f}')
    axes[0, 1].legend()

    # 3. Average Word Count by Category
    avg_words = df_contracts.groupby('category')['word_count'].mean().sort_values()
    axes[1, 0].barh(avg_words.index, avg_words.values, color='seagreen')
    axes[1, 0].set_xlabel('Average Word Count')
    axes[1, 0].set_title('Average Document Length by Category')

    # 4. Structure Metrics Heatmap
    pivot_data = df_contracts.groupby('category')[['table_count', 'section_count', 'paragraph_count']].mean()
    sns.heatmap(pivot_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1, 1])
    axes[1, 1].set_title('Average Structure Metrics by Category')

    plt.tight_layout()
    plt.savefig('contract_eda.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nVisualization saved to contract_eda.png")

# Log EDA results
eda_trace.span(
    name="eda-summary",
    input={
        "total_documents": len(df_contracts),
        "total_words": int(df_contracts['word_count'].sum()),
        "avg_word_count": float(df_contracts['word_count'].mean()),
        "categories": len(df_contracts['category'].unique())
    }
)

langfuse.flush()

## Key Concept: Why EDA Matters for AI Systems

EDA isn't just for traditional ML - it's critical for LLM applications:

| EDA Finding | AI System Implication |
|-------------|----------------------|
| Long documents (>8000 words) | Need chunking strategy for context limits |
| Many tables | Must preserve table structure in embeddings |
| Varied section counts | Section-based retrieval may be inconsistent |
| Category imbalance | Some agents may have less training data |

**Our EDA reveals:**
- Document length distribution for chunking decisions
- Structure complexity for parsing strategies
- Category distribution for balanced training

## 2.4 Contract Entity Extraction

We extract contract-specific entities using regex patterns and domain knowledge.

In [ ]:
# ============================================================================
# WEEK 2.4: CONTRACT ENTITY EXTRACTOR
# ============================================================================

class ContractEntityExtractor:
    """
    Extract contract-specific entities using pattern matching.

    Entities extracted:
    - Parties (organizations involved)
    - Dates (effective, termination, etc.)
    - Monetary values
    - Percentages
    - Durations
    - Legal terms and risk indicators
    """

    def __init__(self):
        # Define extraction patterns
        self.patterns = {
            'parties': r'(?:between|by and between)\s+([A-Z][A-Za-z\s]+(?:Inc\.|LLC|Ltd\.|Corp\.)?)',
            'effective_date': r'(?:effective\s+(?:as\s+of\s+)?(?:date)?:?\s*)(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|[A-Z][a-z]+\s+\d{1,2},?\s+\d{4})',
            'monetary_values': r'\$([\d,]+(?:\.\d{2})?)|(?:USD|INR)\s*([\d,]+(?:\.\d{2})?)',
            'percentages': r'(\d+(?:\.\d+)?\s*%)',
            'durations': r'(\d+)\s*(?:years?|months?|days?|weeks?)',
            'notice_periods': r'(?:notice\s+(?:period)?\s*(?:of)?\s*)(\d+)\s*(?:days?|business\s+days?)',
            'liability_caps': r'(?:liability|cap|limit)\s*(?:shall)?\s*(?:not)?\s*(?:exceed)?\s*\$?([\d,]+)',
            'termination_clauses': r'(?:terminat(?:e|ion))\s*(?:for)?\s*(?:convenience|cause|breach)',
            'indemnification': r'(?:indemnif(?:y|ication)|hold\s+harmless)',
            'confidentiality': r'(?:confidential(?:ity)?|non-disclosure|NDA)',
            'ip_rights': r'(?:intellectual\s+property|IP\s+rights|patent|copyright|trademark)'
        }

        # Risk indicator categories
        self.risk_indicators = {
            'high_risk': [
                'unlimited liability', 'indemnify', 'consequential damages',
                'punitive damages', 'sole discretion', 'unilateral',
                'automatic renewal', 'exclusive remedy', 'waiver'
            ],
            'medium_risk': [
                'limitation of liability', 'cap', 'terminate for convenience',
                'force majeure', 'dispute resolution', 'arbitration'
            ],
            'favorable': [
                'mutual indemnification', 'reasonable efforts', 'good faith',
                'cure period', 'written consent', 'pro-rata refund'
            ]
        }

    def extract_entities(self, text: str) -> Dict[str, List]:
        """Extract all entities from text using patterns."""
        results = {}
        for entity_type, pattern in self.patterns.items():
            matches = re.findall(pattern, text, re.IGNORECASE)
            # Flatten tuple matches
            flat_matches = []
            for m in matches:
                if isinstance(m, tuple):
                    flat_matches.extend([x for x in m if x])
                else:
                    flat_matches.append(m)
            results[entity_type] = list(set(flat_matches))  # Deduplicate
        return results

    def assess_risk_indicators(self, text: str) -> Dict[str, List[str]]:
        """Identify risk indicators in text."""
        text_lower = text.lower()
        found = {'high_risk': [], 'medium_risk': [], 'favorable': []}

        for risk_level, indicators in self.risk_indicators.items():
            for indicator in indicators:
                if indicator.lower() in text_lower:
                    found[risk_level].append(indicator)

        return found

    def process_document(self, doc: Dict, trace_parent=None) -> Dict:
        """Extract all entities from a document with tracing."""
        span = trace_parent.span(name=f"extract-{doc.get('doc_id', 'unknown')}") if trace_parent else None

        # Combine all text sources
        text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')

        entities = self.extract_entities(text)
        risk_indicators = self.assess_risk_indicators(text)

        result = {
            'doc_id': doc.get('doc_id'),
            'category': doc.get('category'),
            'entities': entities,
            'risk_indicators': risk_indicators,
            'entity_counts': {k: len(v) for k, v in entities.items()},
            'risk_counts': {k: len(v) for k, v in risk_indicators.items()}
        }

        if span:
            span.end(output={
                'entity_counts': result['entity_counts'],
                'risk_counts': result['risk_counts']
            })

        return result

print("ContractEntityExtractor class defined")

In [ ]:
# ============================================================================
# EXAMPLE: Entity Extraction on Sample Text
# ============================================================================

# Initialize extractor
entity_extractor = ContractEntityExtractor()

# Sample contract text for demonstration
sample_contract_text = """
MASTER SERVICES AGREEMENT

This Master Services Agreement ("Agreement") is entered into as of January 15, 2024,
by and between TechCorp Inc. ("Client") and ConsultingPro LLC ("Contractor").

1. TERM AND TERMINATION
This Agreement shall be effective for a period of 24 months from the Effective Date.
Either party may terminate for convenience with 90 days written notice.
Either party may terminate for cause upon 30 days notice if the other party breaches.

2. FEES AND PAYMENT
Client shall pay Contractor $150,000 per month for services rendered.
Payment terms are Net 30 from invoice date.
Late payments shall accrue interest at 1.5% per month.

3. LIABILITY
Contractor's total liability shall not exceed $500,000.
Neither party shall be liable for consequential damages.
Contractor shall indemnify Client against third-party claims.

4. CONFIDENTIALITY
Both parties agree to maintain confidentiality of proprietary information.
This NDA provision survives termination for 3 years.

5. INTELLECTUAL PROPERTY
All intellectual property created shall be owned by Client.
Contractor retains rights to pre-existing IP and general knowledge.
"""

print("ENTITY EXTRACTION EXAMPLE")
print("="*60)

# Extract entities
entities = entity_extractor.extract_entities(sample_contract_text)
risks = entity_extractor.assess_risk_indicators(sample_contract_text)

print("\nExtracted Entities:")
for entity_type, values in entities.items():
    if values:
        print(f"  {entity_type}: {values}")

print("\nRisk Indicators:")
for risk_level, indicators in risks.items():
    if indicators:
        print(f"  {risk_level}: {indicators}")

In [ ]:
# ============================================================================
# BATCH ENTITY EXTRACTION WITH TRACING
# ============================================================================

print("\nBATCH ENTITY EXTRACTION")
print("="*60)

# Create trace for batch extraction
extraction_trace = langfuse.trace(
    name="entity-extraction-batch",
    session_id=SESSION_ID,
    input={"document_count": len(all_docs_flat)},
    tags=["entity-extraction", "batch"]
)

all_entity_results = []
for doc in all_docs_flat:
    result = entity_extractor.process_document(doc, extraction_trace)
    all_entity_results.append(result)

extraction_trace.update(output={"extracted_count": len(all_entity_results)})

# Display examples
print("\nExtraction Results (First 3 Documents):")
for result in all_entity_results[:3]:
    print(f"\nDocument: {result['doc_id']} ({result['category']})")
    print(f"  Monetary Values: {result['entities'].get('monetary_values', [])[:3]}")
    print(f"  Durations: {result['entities'].get('durations', [])[:3]}")
    print(f"  Risk Indicators:")
    print(f"    High Risk: {result['risk_counts']['high_risk']}")
    print(f"    Medium Risk: {result['risk_counts']['medium_risk']}")
    print(f"    Favorable: {result['risk_counts']['favorable']}")

langfuse.flush()

## Checkpoint: Week 2 Verification

In [ ]:
# ============================================================================
# WEEK 2 CHECKPOINT
# ============================================================================

print("WEEK 2 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Document processor initialized", doc_processor is not None),
    ("Documents processed", len(all_docs_flat) > 0),
    ("EDA DataFrame created", not df_contracts.empty),
    ("Entity extractor initialized", entity_extractor is not None),
    ("Entity extraction complete", len(all_entity_results) > 0),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("All Week 2 checkpoints PASSED! Ready for Week 3.")
else:
    print("Some checkpoints FAILED. Review the issues above.")

## Week 2 Summary

### What We Accomplished

| Component | Status | Key Learnings |
|-----------|--------|---------------|
| Document Processor | Complete | Section extraction, table handling |
| EDA | Complete | Corpus statistics, visualizations |
| Entity Extractor | Complete | Pattern matching, risk indicators |

### Key Takeaways

1. **Structure Matters**: Contracts have predictable structure we can exploit
2. **Tables are Data**: Table content often contains key terms (pricing, schedules)
3. **Risk is Linguistic**: Certain phrases indicate risk levels
4. **EDA Informs Design**: Document statistics guide chunking and retrieval strategies

### Looking Ahead to Week 3

In Week 3, we will:
- Set up ChromaDB for vector storage
- Create embeddings for all contract chunks
- Implement semantic search
- Build the foundation for RAG pipelines

---
# WEEK 3: ChromaDB Vector Store & Embeddings (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 3: ChromaDB Vector Store & Embeddings

---

## Learning Objectives

By the end of Week 3, you will be able to:

- [ ] Understand how embeddings represent semantic meaning
- [ ] Set up ChromaDB with persistent storage
- [ ] Implement document chunking strategies
- [ ] Create a traced vector store with full observability
- [ ] Perform semantic search across the contract corpus

---

## Key Concept: Embeddings and Semantic Search

**What are embeddings?**

Embeddings are dense vector representations of text where semantic similarity is captured by geometric proximity. Two sentences with similar meanings will have embeddings that are close together in vector space.

```
"The contractor shall indemnify the client"
    ↓ Embedding Model
[0.023, -0.156, 0.089, ..., 0.042]  (1536 dimensions)

"The vendor will hold harmless the customer"
    ↓ Embedding Model
[0.025, -0.152, 0.091, ..., 0.039]  (Similar vector!)
```

**Why this matters for contracts:**
- Legal language varies ("shall" vs "will", "contractor" vs "vendor")
- Keyword search fails on paraphrased clauses
- Semantic search finds conceptually similar content regardless of wording

## 3.1 ChromaDB Vector Store Class

In [ ]:
# ============================================================================
# WEEK 3.1: CHROMADB VECTOR STORE WITH TRACING
# ============================================================================

import chromadb

class ContractVectorStore:
    """
    ChromaDB-based vector store with full Langfuse observability.

    Features:
    - Persistent storage for production use
    - Traced embedding generation
    - Configurable chunking strategies
    - Metadata-rich retrieval

    Design Decisions:
    - Use cosine similarity (standard for text embeddings)
    - Chunk overlap to preserve context at boundaries
    - Store source metadata for provenance tracking
    """

    def __init__(self, persist_directory: str = "./chroma_contracts_db"):
        """
        Initialize ChromaDB with persistent storage.

        Args:
            persist_directory: Path for database persistence
        """
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_model = "text-embedding-3-small"
        self.collections = {}

        # Log initialization
        langfuse.trace(
            name="vectorstore-init",
            session_id=SESSION_ID,
            input={
                "persist_directory": persist_directory,
                "model": self.embedding_model
            }
        )

        print(f"ChromaDB initialized")
        print(f"  Persist directory: {persist_directory}")
        print(f"  Embedding model: {self.embedding_model}")

    def get_embedding_traced(self, text: str, doc_id: str = "unknown") -> List[float]:
        """
        Generate embedding with Langfuse tracing.

        Truncates text to respect model token limits.
        """
        # Truncate to ~8000 chars (roughly 2000 tokens)
        return traced_embedding(text[:8000], trace_name=f"embed-{doc_id}")

    def create_collection(self, name: str) -> chromadb.Collection:
        """
        Create or get a collection with cosine similarity.
        """
        collection = self.client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )
        self.collections[name] = collection
        return collection

    def chunk_contract(self, text: str, chunk_size: int = 800, overlap: int = 100) -> List[str]:
        """
        Split document into overlapping chunks.

        Args:
            text: Full document text
            chunk_size: Target words per chunk
            overlap: Words of overlap between chunks

        Returns:
            List of text chunks

        Why overlap?
        - Prevents information loss at chunk boundaries
        - Improves retrieval of clauses that span chunks
        """
        words = text.split()
        chunks = []

        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk.strip() and len(chunk) > 50:  # Skip tiny chunks
                chunks.append(chunk)

        return chunks if chunks else [text]  # Return original if no chunks

print("ContractVectorStore class defined (Part 1)")

In [ ]:
# ============================================================================
# VECTOR STORE: ADD DOCUMENTS METHOD
# ============================================================================

def add_documents(self, collection_name: str, documents: List[Dict]):
    """
    Add documents to vector store with full tracing.

    Args:
        collection_name: Name of the collection to add to
        documents: List of document dictionaries

    Each document is:
    1. Chunked into manageable pieces
    2. Embedded using traced_embedding()
    3. Stored with rich metadata
    """
    trace = langfuse.trace(
        name=f"index-{collection_name}",
        session_id=SESSION_ID,
        input={"collection": collection_name, "doc_count": len(documents)},
        tags=["indexing", "embeddings"]
    )

    collection = self.create_collection(collection_name)

    all_ids = []
    all_embeddings = []
    all_texts = []
    all_metadata = []

    for doc in documents:
        # Combine text sources
        text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')
        chunks = self.chunk_contract(text)

        doc_id = doc.get('doc_id', 'unknown')
        span = trace.span(
            name=f"embed-{doc_id}",
            input={"chunks": len(chunks)}
        )

        for i, chunk in enumerate(chunks):
            chunk_id = f"{doc_id}_chunk_{i}"
            embedding = self.get_embedding_traced(chunk, doc_id)

            all_ids.append(chunk_id)
            all_embeddings.append(embedding)
            all_texts.append(chunk)
            all_metadata.append({
                'doc_id': doc_id,
                'filename': doc.get('filename', 'unknown'),
                'category': doc.get('category', 'unknown'),
                'chunk_index': i,
                'total_chunks': len(chunks)
            })

        span.end(output={"embedded_chunks": len(chunks)})

    # Batch add to collection
    if all_ids:
        collection.add(
            ids=all_ids,
            embeddings=all_embeddings,
            documents=all_texts,
            metadatas=all_metadata
        )

    trace.update(output={"total_chunks": len(all_ids)})
    print(f"Added {len(all_ids)} chunks to '{collection_name}' collection")

# Add method to class
ContractVectorStore.add_documents = add_documents

print("add_documents() method added")

In [ ]:
# ============================================================================
# VECTOR STORE: SEARCH METHOD
# ============================================================================

def search_traced(self, collection_name: str, query: str, n_results: int = 5) -> Dict:
    """
    Semantic search with full Langfuse tracing.

    Args:
        collection_name: Collection to search
        query: Natural language search query
        n_results: Number of results to return

    Returns:
        Dictionary with documents, metadatas, and distances
    """
    trace = langfuse.trace(
        name="semantic-search",
        session_id=SESSION_ID,
        input={
            "query": query,
            "collection": collection_name,
            "n_results": n_results
        },
        tags=["search", "retrieval"]
    )

    # Ensure collection exists
    if collection_name not in self.collections:
        self.create_collection(collection_name)

    collection = self.collections[collection_name]

    # Generate query embedding
    query_embedding = traced_embedding(query, "search-query")

    # Perform search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )

    # Calculate top similarity score
    top_similarity = 0
    if results['distances'] and results['distances'][0]:
        top_similarity = 1 - results['distances'][0][0]  # Convert distance to similarity

    trace.update(output={
        "results_count": len(results['documents'][0]) if results['documents'] else 0,
        "top_similarity": round(top_similarity, 4)
    })

    return results

# Add method to class
ContractVectorStore.search_traced = search_traced

print("search_traced() method added")

## 3.2 Initialize Vector Store and Index Documents

In [ ]:
# ============================================================================
# WEEK 3.2: INITIALIZE AND INDEX
# ============================================================================

# Initialize vector store
vector_store = ContractVectorStore()

print("\n" + "="*60)
print("INDEXING CONTRACT DOCUMENTS")
print("="*60)
print("Each embedding is traced in Langfuse.\n")

# Index all documents into unified collection
if all_docs_flat:
    vector_store.add_documents("contracts_all", all_docs_flat)
else:
    print("No documents to index - using sample data...")
    # Create sample documents for demonstration
    sample_docs = [
        {
            'doc_id': 'SAMPLE-001',
            'filename': 'sample_msa.docx',
            'category': 'master_agreements',
            'full_text': sample_contract_text,
            'table_text': ''
        }
    ]
    vector_store.add_documents("contracts_all", sample_docs)
    all_docs_flat = sample_docs

print("\nIndexing complete!")
langfuse.flush()

## 3.3 Semantic Search Examples

Let's demonstrate semantic search with multiple query types.

In [ ]:
# ============================================================================
# WEEK 3.3: SEMANTIC SEARCH EXAMPLES
# ============================================================================

test_queries = [
    # Legal queries
    "What are the liability limitations in the contract?",
    "Find indemnification clauses and hold harmless provisions",

    # Financial queries
    "What are the payment terms and billing schedules?",
    "Late payment penalties and interest rates",

    # Operational queries
    "Termination for convenience rights",
    "Notice period requirements",

    # IP and Confidentiality queries
    "Intellectual property ownership and licensing",
    "Data protection and confidentiality requirements"
]

print("SEMANTIC SEARCH DEMONSTRATION")
print("="*70)
print("Each search is traced in Langfuse.\n")

In [ ]:
# ============================================================================
# EXECUTE SEARCHES
# ============================================================================

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}] '{query}'")
    print("-" * 60)

    results = vector_store.search_traced("contracts_all", query, n_results=2)

    if results['documents'] and results['documents'][0]:
        for j, (doc, meta, dist) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )):
            similarity = 1 - dist
            print(f"\n  Result {j+1} (Similarity: {similarity:.3f})")
            print(f"  Source: {meta['filename']} ({meta['category']})")
            print(f"  Preview: {doc[:150]}...")
    else:
        print("  No results found")

langfuse.flush()

## Deep Dive: Understanding Similarity Scores

ChromaDB uses cosine distance, which we convert to similarity:

| Similarity | Interpretation | Action |
|------------|----------------|--------|
| > 0.85 | Very high match | Direct answer likely |
| 0.70 - 0.85 | Good match | Relevant context |
| 0.55 - 0.70 | Moderate match | May need verification |
| < 0.55 | Weak match | Consider rephrasing query |

**Factors affecting similarity:**
1. **Vocabulary overlap**: Similar terms increase similarity
2. **Semantic relatedness**: Related concepts cluster together
3. **Domain specificity**: Legal jargon clusters with legal content
4. **Chunk quality**: Well-formed chunks embed better

## Checkpoint: Week 3 Verification

In [ ]:
# ============================================================================
# WEEK 3 CHECKPOINT
# ============================================================================

print("WEEK 3 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Vector store initialized", vector_store is not None),
    ("Collection created", 'contracts_all' in vector_store.collections),
    ("Documents indexed", len(all_docs_flat) > 0),
    ("Search function works", callable(vector_store.search_traced)),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("All Week 3 checkpoints PASSED! Ready for Week 4.")
else:
    print("Some checkpoints FAILED. Review the issues above.")

## Week 3 Summary

### What We Accomplished

| Component | Status | Langfuse Traces |
|-----------|--------|----------------|
| ChromaDB Setup | Complete | vectorstore-init |
| Document Chunking | Complete | (within indexing) |
| Embedding Generation | Complete | embed-* |
| Indexing Pipeline | Complete | index-contracts_all |
| Semantic Search | Complete | semantic-search |

### Key Takeaways

1. **Embeddings capture meaning**: Semantic search finds related content regardless of exact wording
2. **Chunking strategy matters**: Overlap preserves context at boundaries
3. **Metadata enables filtering**: Store rich metadata for provenance tracking
4. **Tracing is essential**: Every embedding call should be observable

### Looking Ahead to Week 4

In Week 4, we will:
- Design specialized risk assessment agents
- Implement structured output parsing with Pydantic
- Create Legal, Financial, and Operational agents
- Integrate LangChain LCEL for agent chains

---
# WEEK 4: Single Agent Design - Risk Assessment Agents (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 4: Single Agent Design - Risk Assessment Agents

---

## Learning Objectives

By the end of Week 4, you will be able to:

- [ ] Design structured output schemas using Pydantic
- [ ] Create specialized risk assessment agents
- [ ] Implement LangChain LCEL (LangChain Expression Language) chains
- [ ] Integrate Langfuse tracing with LangChain
- [ ] Run and evaluate multiple agent invocations

---

## Key Concept: Why Structured Output?

LLMs naturally produce unstructured text, but applications need structured data:

```
Unstructured (hard to process):
"The contract has medium risk due to the liability clause..."

Structured (programmatically useful):
{
    "risk_level": "MEDIUM",
    "risk_factors": ["unlimited liability", "no cap"],
    "confidence": 0.85,
    "recommended_actions": ["Add liability cap", "Negotiate indemnification"]
}
```

**Pydantic** provides:
- Type validation for LLM outputs
- Automatic JSON schema generation
- Clear error messages when parsing fails
- Documentation through field descriptions

## 4.1 Pydantic Models for Structured Output

We define Pydantic models that specify the exact structure we expect from each agent.

In [ ]:
# ============================================================================
# WEEK 4.1: PYDANTIC MODELS FOR STRUCTURED OUTPUTS
# ============================================================================

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
try:
    from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
except (ImportError, ModuleNotFoundError):
    try:
        from langfuse.langchain import CallbackHandler as LangfuseCallbackHandler
    except (ImportError, ModuleNotFoundError):
        # Langfuse v3+: CallbackHandler not needed, create a no-op
        class LangfuseCallbackHandler:
            def __init__(self, **kwargs): pass

class LegalRiskAssessment(BaseModel):
    """
    Structured output for legal risk analysis.

    Each field has a description that guides the LLM's output.
    The Field() function adds constraints and documentation.
    """
    contract_type: str = Field(
        description="Type of contract (MSA, NDA, SOW, etc.)"
    )
    risk_level: str = Field(
        description="Overall risk level: LOW, MEDIUM, HIGH, CRITICAL"
    )
    liability_exposure: str = Field(
        description="Assessment of liability exposure and caps"
    )
    ip_risks: List[str] = Field(
        description="Intellectual property related risks identified"
    )
    indemnification_issues: List[str] = Field(
        description="Indemnification clause concerns and imbalances"
    )
    compliance_gaps: List[str] = Field(
        description="Regulatory compliance gaps identified"
    )
    recommended_changes: List[str] = Field(
        description="Specific recommended contract modifications"
    )
    negotiation_leverage: str = Field(
        description="Areas where negotiation leverage exists"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Assessment confidence score (0.0 to 1.0)"
    )

print("LegalRiskAssessment model defined")
print(f"  Fields: {list(LegalRiskAssessment.model_fields.keys())}")

In [ ]:
# ============================================================================
# FINANCIAL RISK ASSESSMENT MODEL
# ============================================================================

class FinancialRiskAssessment(BaseModel):
    """
    Structured output for financial risk analysis.

    Focuses on monetary aspects of contracts:
    - Pricing and payment terms
    - Financial exposure limits
    - Cost escalation risks
    """
    total_contract_value: str = Field(
        description="Total estimated contract value or range"
    )
    payment_terms: str = Field(
        description="Summary of payment terms (Net 30, milestones, etc.)"
    )
    pricing_risks: List[str] = Field(
        description="Pricing-related risks (escalation clauses, forex, etc.)"
    )
    penalty_clauses: List[str] = Field(
        description="Financial penalties identified in contract"
    )
    cash_flow_impact: str = Field(
        description="Impact on cash flow (positive/negative/neutral)"
    )
    cost_escalation_risk: str = Field(
        description="Risk of cost escalation (LOW/MEDIUM/HIGH)"
    )
    financial_exposure: str = Field(
        description="Maximum financial exposure amount or description"
    )
    recommended_caps: List[str] = Field(
        description="Recommended financial caps or limits"
    )
    risk_level: str = Field(
        description="Overall financial risk level: LOW, MEDIUM, HIGH, CRITICAL"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Assessment confidence score"
    )

print("FinancialRiskAssessment model defined")

In [ ]:
# ============================================================================
# OPERATIONAL RISK ASSESSMENT MODEL
# ============================================================================

class OperationalRiskAssessment(BaseModel):
    """
    Structured output for operational risk analysis.

    Focuses on execution and delivery aspects:
    - Service scope and SLAs
    - Resource requirements
    - Delivery risks
    """
    service_scope: str = Field(
        description="Summary of service scope and deliverables"
    )
    sla_requirements: List[str] = Field(
        description="SLA requirements identified (uptime, response time, etc.)"
    )
    delivery_risks: List[str] = Field(
        description="Delivery and timeline risks identified"
    )
    resource_requirements: str = Field(
        description="Resource requirements assessment"
    )
    dependency_risks: List[str] = Field(
        description="Third-party and internal dependencies"
    )
    performance_metrics: List[str] = Field(
        description="Key performance metrics defined in contract"
    )
    operational_gaps: List[str] = Field(
        description="Operational capability gaps identified"
    )
    mitigation_strategies: List[str] = Field(
        description="Risk mitigation strategies recommended"
    )
    risk_level: str = Field(
        description="Overall operational risk level: LOW, MEDIUM, HIGH, CRITICAL"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Assessment confidence score"
    )

print("OperationalRiskAssessment model defined")
print("\nAll three Pydantic models ready for agent implementation.")

## 4.2 Risk Assessment Agents with Langfuse Integration

Now we create the actual agent classes that use these Pydantic models.

In [ ]:
# ============================================================================
# WEEK 4.2: LANGFUSE CALLBACK HANDLER FACTORY
# ============================================================================

def get_langfuse_handler(trace_name: str, tags: List[str] = None) -> LangfuseCallbackHandler:
    """
    Create a Langfuse callback handler for LangChain integration.

    This handler automatically traces:
    - Prompt formatting
    - LLM calls (input/output/tokens)
    - Output parsing
    - Errors and retries

    Args:
        trace_name: Identifier for this trace
        tags: Optional tags for filtering in Langfuse

    Returns:
        Configured LangfuseCallbackHandler
    """
    return LangfuseCallbackHandler(
        secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
        public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
        host=os.environ.get('LANGFUSE_HOST'),
        session_id=SESSION_ID,
        trace_name=trace_name,
        tags=tags or []
    )

print("get_langfuse_handler() factory function defined")

In [ ]:
# ============================================================================
# LEGAL RISK AGENT
# ============================================================================

class LegalRiskAgent:
    """
    Legal Risk Assessment Agent with full Langfuse tracing.

    This agent specializes in:
    - Liability analysis
    - IP rights assessment
    - Indemnification review
    - Compliance gap identification

    Architecture:
    1. Prompt template defines the expert persona
    2. PydanticOutputParser ensures structured output
    3. LangChain LCEL chains components together
    4. Langfuse callback traces everything
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=LegalRiskAssessment)

        # Define the expert prompt
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert contract attorney specializing in enterprise agreements.

Your expertise includes:
- Liability analysis and limitation clauses
- Intellectual property rights and protections
- Indemnification provisions and fairness assessment
- Regulatory compliance requirements
- Contract negotiation strategy

Analyze contracts thoroughly and identify legal risks with specific citations.

{format_instructions}"""),
            ("human", """Analyze the following contract for legal risks:

{contract_text}

Provide a comprehensive legal risk assessment.""")
        ])

    def analyze(self, contract_text: str, doc_id: str = "unknown") -> LegalRiskAssessment:
        """
        Analyze contract with full Langfuse tracing.

        Args:
            contract_text: The contract text to analyze
            doc_id: Document identifier for tracing

        Returns:
            LegalRiskAssessment structured output
        """
        # Create traced handler
        handler = get_langfuse_handler(
            trace_name=f"legal-agent-{doc_id}",
            tags=["legal", "risk-assessment", "agent"]
        )

        # Create LLM with tracing
        llm = ChatOpenAI(
            model=self.model,
            temperature=0,  # Deterministic for consistency
            callbacks=[handler]
        )

        # Build LCEL chain: prompt -> llm -> parser
        chain = self.prompt | llm | self.parser

        # Execute chain
        return chain.invoke({
            "contract_text": contract_text[:8000],  # Respect context limits
            "format_instructions": self.parser.get_format_instructions()
        })

print("LegalRiskAgent class defined")

In [ ]:
# ============================================================================
# FINANCIAL RISK AGENT
# ============================================================================

class FinancialRiskAgent:
    """
    Financial Risk Assessment Agent with full Langfuse tracing.

    This agent specializes in:
    - Pricing structure analysis
    - Payment terms evaluation
    - Financial exposure calculation
    - Cost risk identification
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=FinancialRiskAssessment)

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert financial analyst specializing in contract economics.

Your expertise includes:
- Pricing structure and total contract value analysis
- Payment terms and cash flow implications
- Penalty clauses and financial exposure
- Cost escalation risk assessment
- Budget and financial planning

Analyze contracts from a financial perspective with specific monetary findings.

{format_instructions}"""),
            ("human", """Analyze the following contract for financial risks:

{contract_text}

Provide a comprehensive financial risk assessment.""")
        ])

    def analyze(self, contract_text: str, doc_id: str = "unknown") -> FinancialRiskAssessment:
        """Analyze contract with Langfuse tracing."""
        handler = get_langfuse_handler(
            trace_name=f"financial-agent-{doc_id}",
            tags=["financial", "risk-assessment", "agent"]
        )

        llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
        chain = self.prompt | llm | self.parser

        return chain.invoke({
            "contract_text": contract_text[:8000],
            "format_instructions": self.parser.get_format_instructions()
        })

print("FinancialRiskAgent class defined")

In [ ]:
# ============================================================================
# OPERATIONAL RISK AGENT
# ============================================================================

class OperationalRiskAgent:
    """
    Operational Risk Assessment Agent with full Langfuse tracing.

    This agent specializes in:
    - Service scope clarity
    - SLA requirements analysis
    - Delivery timeline risks
    - Resource and dependency assessment
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=OperationalRiskAssessment)

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert operations manager specializing in service delivery.

Your expertise includes:
- Service scope clarity and completeness
- SLA requirements and achievability
- Delivery timelines and milestones
- Resource planning and capacity
- Dependency management

Analyze contracts from an operational perspective with practical insights.

{format_instructions}"""),
            ("human", """Analyze the following contract for operational risks:

{contract_text}

Provide a comprehensive operational risk assessment.""")
        ])

    def analyze(self, contract_text: str, doc_id: str = "unknown") -> OperationalRiskAssessment:
        """Analyze contract with Langfuse tracing."""
        handler = get_langfuse_handler(
            trace_name=f"operational-agent-{doc_id}",
            tags=["operational", "risk-assessment", "agent"]
        )

        llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
        chain = self.prompt | llm | self.parser

        return chain.invoke({
            "contract_text": contract_text[:8000],
            "format_instructions": self.parser.get_format_instructions()
        })

print("OperationalRiskAgent class defined")

In [ ]:
# ============================================================================
# INITIALIZE ALL AGENTS
# ============================================================================

# Create agent instances
legal_agent = LegalRiskAgent()
financial_agent = FinancialRiskAgent()
operational_agent = OperationalRiskAgent()

print("Risk Assessment Agents Initialized:")
print("  - LegalRiskAgent (liability, IP, indemnification)")
print("  - FinancialRiskAgent (pricing, payment, exposure)")
print("  - OperationalRiskAgent (scope, SLA, delivery)")
print("\nAll agents use Langfuse tracing via LangChain callbacks.")

## 4.3 Agent Invocation Examples

Let's run each agent and observe the structured outputs.

In [ ]:
# ============================================================================
# EXAMPLE 1: LEGAL AGENT ANALYSIS
# ============================================================================

print("EXAMPLE 1: Legal Risk Agent")
print("="*60)

try:
    legal_result = legal_agent.analyze(sample_contract_text, "example-001")

    print(f"\nContract Type: {legal_result.contract_type}")
    print(f"Risk Level: {legal_result.risk_level}")
    print(f"Confidence: {legal_result.confidence:.2f}")
    print(f"\nLiability Exposure:")
    print(f"  {legal_result.liability_exposure[:150]}...")
    print(f"\nIP Risks: {legal_result.ip_risks[:3]}")
    print(f"\nRecommended Changes:")
    for i, change in enumerate(legal_result.recommended_changes[:3], 1):
        print(f"  {i}. {change}")
except Exception as e:
    print(f"Error: {e}")

langfuse.flush()

In [ ]:
# ============================================================================
# EXAMPLE 2: FINANCIAL AGENT ANALYSIS
# ============================================================================

print("\nEXAMPLE 2: Financial Risk Agent")
print("="*60)

try:
    financial_result = financial_agent.analyze(sample_contract_text, "example-001")

    print(f"\nContract Value: {financial_result.total_contract_value}")
    print(f"Payment Terms: {financial_result.payment_terms}")
    print(f"Risk Level: {financial_result.risk_level}")
    print(f"Confidence: {financial_result.confidence:.2f}")
    print(f"\nPricing Risks:")
    for risk in financial_result.pricing_risks[:3]:
        print(f"  - {risk}")
    print(f"\nFinancial Exposure: {financial_result.financial_exposure}")
except Exception as e:
    print(f"Error: {e}")

langfuse.flush()

In [ ]:
# ============================================================================
# EXAMPLE 3: OPERATIONAL AGENT ANALYSIS
# ============================================================================

print("\nEXAMPLE 3: Operational Risk Agent")
print("="*60)

try:
    operational_result = operational_agent.analyze(sample_contract_text, "example-001")

    print(f"\nService Scope: {operational_result.service_scope[:150]}...")
    print(f"Risk Level: {operational_result.risk_level}")
    print(f"Confidence: {operational_result.confidence:.2f}")
    print(f"\nSLA Requirements:")
    for sla in operational_result.sla_requirements[:3]:
        print(f"  - {sla}")
    print(f"\nDelivery Risks:")
    for risk in operational_result.delivery_risks[:3]:
        print(f"  - {risk}")
except Exception as e:
    print(f"Error: {e}")

langfuse.flush()

## Checkpoint: Week 4 Verification

In [ ]:
# ============================================================================
# WEEK 4 CHECKPOINT
# ============================================================================

print("WEEK 4 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Pydantic models defined", LegalRiskAssessment is not None),
    ("Legal agent initialized", legal_agent is not None),
    ("Financial agent initialized", financial_agent is not None),
    ("Operational agent initialized", operational_agent is not None),
    ("Langfuse handler factory works", callable(get_langfuse_handler)),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("All Week 4 checkpoints PASSED! Ready for Week 5.")
else:
    print("Some checkpoints FAILED. Review the issues above.")

## Week 4 Summary

### What We Accomplished

| Component | Status | Langfuse Traces |
|-----------|--------|----------------|
| Pydantic Models | 3 models | N/A |
| Legal Agent | Complete | legal-agent-* |
| Financial Agent | Complete | financial-agent-* |
| Operational Agent | Complete | operational-agent-* |

### Key Takeaways

1. **Structured Output**: Pydantic ensures consistent, typed agent outputs
2. **Expert Prompts**: System prompts define agent expertise and focus
3. **LCEL Chains**: LangChain Expression Language simplifies chain composition
4. **Integrated Tracing**: Langfuse callbacks trace entire chain execution

### Looking Ahead to Week 5

In Week 5, we will:
- Orchestrate multiple agents in parallel
- Implement consensus mechanisms
- Add retry logic and error handling
- Build comprehensive risk assessment workflows

---
# WEEK 5: Multi-Agent Orchestration (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 5: Multi-Agent Orchestration

---

## Learning Objectives

By the end of Week 5, you will be able to:

- [ ] Implement parallel agent execution
- [ ] Build consensus mechanisms across agents
- [ ] Add retry logic with exponential backoff
- [ ] Create comprehensive orchestration workflows
- [ ] Calculate composite risk scores

---

## Key Concept: Why Multi-Agent?

Single agents have expertise but limited perspective. Multi-agent systems provide:

```
Contract Input
      |
      v
+------------------+
|   Orchestrator   |
+------------------+
      |  |  |
      v  v  v
+-----+ +-----+ +-----+
|Legal| |Fin. | |Ops. |
+-----+ +-----+ +-----+
      \   |   /
       \  |  /
        v v v
+------------------+
|    Consensus     |
+------------------+
         |
         v
  Combined Assessment
```

**Benefits:**
1. **Coverage**: Multiple perspectives reduce blind spots
2. **Validation**: Agents can verify each other's findings
3. **Speed**: Parallel execution reduces latency
4. **Specialization**: Each agent optimized for its domain

In [ ]:
# ============================================================================
# WEEK 5.1: MULTI-AGENT ORCHESTRATOR
# ============================================================================

from tenacity import retry, stop_after_attempt, wait_exponential
import concurrent.futures

class RiskDimension(Enum):
    """Risk dimensions for orchestration."""
    LEGAL = "legal"
    FINANCIAL = "financial"
    OPERATIONAL = "operational"

class ContractRiskOrchestrator:
    """
    Multi-agent orchestrator with comprehensive Langfuse tracing.

    Features:
    - Parallel agent execution using ThreadPoolExecutor
    - Retry logic with exponential backoff
    - Consensus building across agent outputs
    - Composite risk score calculation
    - Full observability through Langfuse
    """

    def __init__(self):
        self.legal_agent = legal_agent
        self.financial_agent = financial_agent
        self.operational_agent = operational_agent
        self.vector_store = vector_store

        # Risk weights for composite scoring
        self.risk_weights = {
            RiskDimension.LEGAL: 0.4,      # Legal carries most weight
            RiskDimension.FINANCIAL: 0.35, # Financial is second
            RiskDimension.OPERATIONAL: 0.25 # Operational is third
        }

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _execute_agent(self, agent_type: RiskDimension, contract_text: str, doc_id: str) -> Dict:
        """
        Execute a single agent with retry logic.

        Uses tenacity for automatic retries with exponential backoff.
        This handles transient API failures gracefully.
        """
        try:
            if agent_type == RiskDimension.LEGAL:
                result = self.legal_agent.analyze(contract_text, doc_id)
            elif agent_type == RiskDimension.FINANCIAL:
                result = self.financial_agent.analyze(contract_text, doc_id)
            elif agent_type == RiskDimension.OPERATIONAL:
                result = self.operational_agent.analyze(contract_text, doc_id)
            else:
                return {'error': f'Unknown agent type: {agent_type}'}

            return {
                'dimension': agent_type.value,
                'success': True,
                'result': result.model_dump()
            }
        except Exception as e:
            return {
                'dimension': agent_type.value,
                'success': False,
                'error': str(e)
            }

print("ContractRiskOrchestrator class defined (Part 1)")

In [ ]:
# ============================================================================
# ORCHESTRATOR: PARALLEL ANALYSIS METHOD
# ============================================================================

def analyze_parallel(self, contract_text: str, doc_id: str = "unknown") -> Dict:
    """
    Run all agents in parallel with comprehensive tracing.

    Args:
        contract_text: The contract to analyze
        doc_id: Document identifier

    Returns:
        Comprehensive assessment with all agent results and consensus
    """
    # Create orchestration trace
    trace = langfuse.trace(
        name=f"orchestration-{doc_id}",
        session_id=SESSION_ID,
        input={"doc_id": doc_id, "text_length": len(contract_text)},
        tags=["orchestration", "multi-agent"]
    )

    results = {
        'doc_id': doc_id,
        'timestamp': datetime.now().isoformat(),
        'assessments': {},
        'consensus': None,
        'overall_risk': None
    }

    # Execute agents in parallel using ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        # Submit all agent tasks
        futures = {
            executor.submit(self._execute_agent, dim, contract_text, doc_id): dim
            for dim in RiskDimension
        }

        # Collect results as they complete
        for future in concurrent.futures.as_completed(futures):
            dimension = futures[future]
            try:
                result = future.result()
                results['assessments'][dimension.value] = result

                # Log to trace
                trace.span(
                    name=f"agent-{dimension.value}",
                    input={"dimension": dimension.value},
                    output={
                        "success": result.get('success'),
                        "risk_level": result.get('result', {}).get('risk_level')
                    }
                )
            except Exception as e:
                results['assessments'][dimension.value] = {
                    'success': False,
                    'error': str(e)
                }

    # Build consensus from agent results
    results['consensus'] = self._build_consensus(results['assessments'])
    results['overall_risk'] = self._calculate_overall_risk(results['consensus'])

    # Update trace with final results
    trace.update(output={
        "overall_risk": results['overall_risk'],
        "risk_summary": results['consensus']['risk_summary'],
        "action_count": len(results['consensus']['priority_actions'])
    })

    return results

# Add method to class
ContractRiskOrchestrator.analyze_parallel = analyze_parallel

print("analyze_parallel() method added")

In [ ]:
# ============================================================================
# ORCHESTRATOR: CONSENSUS BUILDING
# ============================================================================

def _build_consensus(self, assessments: Dict) -> Dict:
    """
    Build consensus from multiple agent assessments.

    Aggregates findings across agents to create unified view.
    """
    consensus = {
        'risk_summary': {},
        'priority_actions': [],
        'all_recommendations': [],
        'confidence_scores': {}
    }

    for dim, assessment in assessments.items():
        if assessment.get('success') and assessment.get('result'):
            result = assessment['result']

            # Collect risk levels
            consensus['risk_summary'][dim] = result.get('risk_level', 'UNKNOWN')
            consensus['confidence_scores'][dim] = result.get('confidence', 0.5)

            # Collect recommendations
            for key in ['recommended_changes', 'recommended_caps', 'mitigation_strategies']:
                if key in result:
                    consensus['all_recommendations'].extend(result[key])

    # Deduplicate and prioritize actions
    consensus['priority_actions'] = list(set(consensus['all_recommendations']))[:10]

    return consensus

def _calculate_overall_risk(self, consensus: Dict) -> str:
    """
    Calculate weighted overall risk from consensus.

    Uses predefined weights for each dimension.
    """
    risk_scores = {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3, 'CRITICAL': 4}
    total_score = 0
    total_weight = 0

    for dim in RiskDimension:
        risk_level = consensus['risk_summary'].get(dim.value, 'MEDIUM')
        score = risk_scores.get(risk_level, 2)
        weight = self.risk_weights[dim]
        total_score += score * weight
        total_weight += weight

    if total_weight > 0:
        avg_score = total_score / total_weight
        if avg_score < 1.5:
            return 'LOW'
        elif avg_score < 2.5:
            return 'MEDIUM'
        elif avg_score < 3.5:
            return 'HIGH'
        else:
            return 'CRITICAL'

    return 'UNKNOWN'

# Add methods to class
ContractRiskOrchestrator._build_consensus = _build_consensus
ContractRiskOrchestrator._calculate_overall_risk = _calculate_overall_risk

print("Consensus building methods added")

In [ ]:
# ============================================================================
# INITIALIZE ORCHESTRATOR
# ============================================================================

orchestrator = ContractRiskOrchestrator()

print("Contract Risk Orchestrator initialized")
print(f"\nRisk Weights:")
for dim, weight in orchestrator.risk_weights.items():
    print(f"  {dim.value}: {weight:.0%}")

## 5.2 Orchestration Examples

In [ ]:
# ============================================================================
# ORCHESTRATION EXAMPLE
# ============================================================================

print("MULTI-AGENT ORCHESTRATION EXAMPLE")
print("="*70)
print("Running all agents in parallel...\n")

# Run orchestration
orch_result = orchestrator.analyze_parallel(sample_contract_text, "orch-example-001")

print(f"OVERALL RISK: {orch_result['overall_risk']}")
print(f"\nRisk by Dimension:")
for dim, risk in orch_result['consensus']['risk_summary'].items():
    confidence = orch_result['consensus']['confidence_scores'].get(dim, 0)
    print(f"  {dim.title()}: {risk} (confidence: {confidence:.2f})")

print(f"\nTop Priority Actions:")
for i, action in enumerate(orch_result['consensus']['priority_actions'][:5], 1):
    print(f"  {i}. {action}")

langfuse.flush()

## Week 5 Summary

### What We Accomplished

| Component | Status | Key Features |
|-----------|--------|-------------|
| Parallel Execution | Complete | ThreadPoolExecutor, 3 workers |
| Retry Logic | Complete | Exponential backoff, 3 attempts |
| Consensus Building | Complete | Risk aggregation, action prioritization |
| Composite Scoring | Complete | Weighted average across dimensions |

### Key Takeaways

1. **Parallel is faster**: 3 agents run simultaneously vs. sequentially
2. **Retry is essential**: Transient failures happen in production
3. **Consensus adds value**: Multiple perspectives reduce bias
4. **Weights matter**: Adjust based on your risk priorities

---
# WEEK 6: Contract Knowledge Graph (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 6: Contract Knowledge Graph

---

## Learning Objectives

By the end of Week 6, you will be able to:

- [ ] Build a knowledge graph using NetworkX
- [ ] Model contract relationships and hierarchies
- [ ] Visualize graphs with matplotlib and PyVis
- [ ] Query the knowledge graph for insights

---

## Key Concept: Knowledge Graphs

A knowledge graph represents entities and their relationships:

```
(MSA) --[GOVERNS]--> (SOW)
(MSA) --[CONTAINS]--> (Liability Clause)
(Liability Clause) --[MITIGATES]--> (Financial Risk)
```

**Benefits for Contract Intelligence:**
- Understand document hierarchies
- Trace risk to source clauses
- Identify missing protections
- Visualize complex relationships

In [ ]:
# ============================================================================
# WEEK 6.1: CONTRACT KNOWLEDGE GRAPH
# ============================================================================

import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt

class ContractKnowledgeGraph:
    """
    Knowledge graph for contract relationships.

    Models:
    - Contract type hierarchies
    - Clause types and their purposes
    - Risk mitigations
    - Document relationships
    """

    def __init__(self):
        self.graph = nx.DiGraph()
        self._build_contract_ontology()

        # Log initialization
        langfuse.trace(
            name="knowledge-graph-init",
            session_id=SESSION_ID,
            input={"type": "contract-ontology"}
        )

    def _build_contract_ontology(self):
        """Build the foundational contract knowledge graph."""

        # Contract type hierarchy
        contract_hierarchy = {
            'MSA': ['SOW', 'NDA', 'RATE_CARD'],
            'SOW': ['INVOICE', 'SERVICE_REPORT', 'CHANGE_ORDER'],
            'NDA': ['CONFIDENTIAL_INFO'],
            'RATE_CARD': ['PRICING_TABLE']
        }

        for contract_type, subordinates in contract_hierarchy.items():
            self.graph.add_node(contract_type, type='CONTRACT_TYPE', color='#3498db')
            for sub in subordinates:
                self.graph.add_node(sub, type='CONTRACT_TYPE', color='#3498db')
                self.graph.add_edge(contract_type, sub, relation='GOVERNS')

        # Clause types and their concerns
        clause_types = [
            ('INDEMNIFICATION', ['liability_exposure', 'legal_cost', 'third_party_claims']),
            ('LIABILITY_LIMITATION', ['financial_cap', 'damage_types', 'exclusions']),
            ('CONFIDENTIALITY', ['data_protection', 'trade_secret', 'disclosure_rules']),
            ('TERMINATION', ['exit_rights', 'cure_period', 'notice_requirements']),
            ('IP_RIGHTS', ['ownership', 'licensing', 'work_product']),
            ('PAYMENT_TERMS', ['billing_cycle', 'late_fees', 'currency'])
        ]

        for clause, concerns in clause_types:
            self.graph.add_node(clause, type='CLAUSE_TYPE', color='#e74c3c')
            for concern in concerns:
                self.graph.add_node(concern, type='CONCERN', color='#f39c12')
                self.graph.add_edge(clause, concern, relation='ADDRESSES')

        # Risk mitigations
        risk_mitigations = {
            'UNLIMITED_LIABILITY': ['LIABILITY_LIMITATION', 'INDEMNIFICATION'],
            'IP_INFRINGEMENT': ['INDEMNIFICATION', 'IP_RIGHTS'],
            'DATA_BREACH': ['CONFIDENTIALITY', 'INDEMNIFICATION'],
            'PAYMENT_DEFAULT': ['PAYMENT_TERMS', 'TERMINATION'],
            'SCOPE_CREEP': ['TERMINATION', 'PAYMENT_TERMS']
        }

        for risk, clauses in risk_mitigations.items():
            self.graph.add_node(risk, type='RISK', color='#9b59b6')
            for clause in clauses:
                self.graph.add_edge(clause, risk, relation='MITIGATES')

print("ContractKnowledgeGraph class defined (Part 1)")

In [ ]:
# ============================================================================
# KNOWLEDGE GRAPH: QUERY METHODS
# ============================================================================

def add_contract(self, doc: Dict):
    """Add a contract document to the knowledge graph."""
    doc_id = doc.get('doc_id', 'UNKNOWN')
    self.graph.add_node(
        doc_id,
        type='CONTRACT',
        color='#2ecc71',
        filename=doc.get('filename'),
        category=doc.get('category')
    )

    # Link to category
    category = doc.get('category', 'unknown')
    if not self.graph.has_node(category):
        self.graph.add_node(category, type='CATEGORY', color='#1abc9c')
    self.graph.add_edge(doc_id, category, relation='BELONGS_TO')

def get_risk_mitigations(self, risk: str) -> List[str]:
    """Get clauses that mitigate a specific risk."""
    mitigations = []
    if self.graph.has_node(risk):
        for source, target, data in self.graph.in_edges(risk, data=True):
            if data.get('relation') == 'MITIGATES':
                mitigations.append(source)
    return mitigations

def get_clause_concerns(self, clause: str) -> List[str]:
    """Get concerns addressed by a clause type."""
    concerns = []
    if self.graph.has_node(clause):
        for source, target, data in self.graph.out_edges(clause, data=True):
            if data.get('relation') == 'ADDRESSES':
                concerns.append(target)
    return concerns

# Add methods to class
ContractKnowledgeGraph.add_contract = add_contract
ContractKnowledgeGraph.get_risk_mitigations = get_risk_mitigations
ContractKnowledgeGraph.get_clause_concerns = get_clause_concerns

print("Query methods added to knowledge graph")

In [ ]:
# ============================================================================
# KNOWLEDGE GRAPH: VISUALIZATION METHODS
# ============================================================================

def visualize_matplotlib(self):
    """Create matplotlib visualization."""
    plt.figure(figsize=(16, 12))

    colors = [self.graph.nodes[n].get('color', '#95a5a6') for n in self.graph.nodes()]
    pos = nx.spring_layout(self.graph, k=2, iterations=50, seed=42)

    nx.draw_networkx_nodes(self.graph, pos, node_color=colors, node_size=500, alpha=0.9)
    nx.draw_networkx_labels(self.graph, pos, font_size=8)
    nx.draw_networkx_edges(self.graph, pos, edge_color='gray', arrows=True, alpha=0.5)

    plt.title("Contract Knowledge Graph", fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('contract_knowledge_graph.png', dpi=150, bbox_inches='tight')
    plt.show()

def visualize_interactive(self, filename: str = "contract_kg.html"):
    """Create interactive PyVis visualization."""
    net = Network(height="600px", width="100%", directed=True, notebook=True)

    for node, data in self.graph.nodes(data=True):
        net.add_node(
            node,
            label=str(node)[:20],
            color=data.get('color', '#95a5a6'),
            title=f"Type: {data.get('type', 'UNKNOWN')}"
        )

    for source, target, data in self.graph.edges(data=True):
        net.add_edge(source, target, title=data.get('relation', ''))

    net.save_graph(filename)
    print(f"Interactive graph saved to {filename}")
    return filename

# Add methods to class
ContractKnowledgeGraph.visualize_matplotlib = visualize_matplotlib
ContractKnowledgeGraph.visualize_interactive = visualize_interactive

print("Visualization methods added")

In [ ]:
# ============================================================================
# INITIALIZE AND POPULATE KNOWLEDGE GRAPH
# ============================================================================

# Create knowledge graph
contract_kg = ContractKnowledgeGraph()

# Add all processed contracts
for doc in all_docs_flat:
    contract_kg.add_contract(doc)

# Log to Langfuse
langfuse.trace(
    name="knowledge-graph-built",
    session_id=SESSION_ID,
    input={"contracts_added": len(all_docs_flat)},
    output={
        "nodes": contract_kg.graph.number_of_nodes(),
        "edges": contract_kg.graph.number_of_edges()
    }
)

print(f"Contract Knowledge Graph Built:")
print(f"  Nodes: {contract_kg.graph.number_of_nodes()}")
print(f"  Edges: {contract_kg.graph.number_of_edges()}")

In [ ]:
# ============================================================================
# VISUALIZE KNOWLEDGE GRAPH
# ============================================================================

print("KNOWLEDGE GRAPH VISUALIZATION")
print("="*60)

# Create static visualization
contract_kg.visualize_matplotlib()

# Create interactive visualization
contract_kg.visualize_interactive("contract_knowledge_graph.html")

print("\nVisualization files created.")

In [ ]:
# ============================================================================
# KNOWLEDGE GRAPH QUERIES
# ============================================================================

print("\nKNOWLEDGE GRAPH QUERIES")
print("="*60)

# Query 1: Risk mitigations
print("\n1. Risk Mitigation Strategies:")
for risk in ['UNLIMITED_LIABILITY', 'IP_INFRINGEMENT', 'DATA_BREACH']:
    mitigations = contract_kg.get_risk_mitigations(risk)
    print(f"   {risk}: {mitigations}")

# Query 2: Clause concerns
print("\n2. Clause Concerns:")
for clause in ['INDEMNIFICATION', 'CONFIDENTIALITY', 'TERMINATION']:
    concerns = contract_kg.get_clause_concerns(clause)
    print(f"   {clause}: {concerns}")

# Query 3: Node type distribution
print("\n3. Node Type Distribution:")
type_counts = {}
for _, data in contract_kg.graph.nodes(data=True):
    t = data.get('type', 'OTHER')
    type_counts[t] = type_counts.get(t, 0) + 1
for t, count in sorted(type_counts.items()):
    print(f"   {t}: {count}")

langfuse.flush()

## Week 6 Summary

### What We Accomplished

| Component | Status | Key Features |
|-----------|--------|-------------|
| Contract Ontology | Complete | Hierarchy, clauses, risks |
| Document Integration | Complete | Links contracts to categories |
| Query Methods | Complete | Risk mitigations, clause concerns |
| Visualizations | Complete | Static (matplotlib), Interactive (PyVis) |

### Key Takeaways

1. **Relationships matter**: Graphs capture relationships keywords miss
2. **Ontology is foundation**: Domain knowledge encoded in graph structure
3. **Queries enable reasoning**: Find mitigations for any risk
4. **Visualization aids understanding**: Complex relationships become clear

---
# WEEK 7: Advanced Observability & Metrics (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 7: Advanced Observability & Metrics

---

## Learning Objectives

By the end of Week 7, you will be able to:

- [ ] Review and analyze traces in Langfuse
- [ ] Add custom scores to traces
- [ ] Implement quality metrics
- [ ] Create observability dashboards

---

## Key Concept: Production Observability

In production, you need to:

1. **Monitor**: Track system health and performance
2. **Evaluate**: Score output quality over time
3. **Debug**: Trace issues to specific calls
4. **Optimize**: Identify bottlenecks and costs

**Langfuse Metrics:**
- Latency per operation
- Token usage and costs
- Success/failure rates
- Custom quality scores

In [ ]:
# ============================================================================
# WEEK 7.1: OBSERVABILITY SUMMARY
# ============================================================================

print("LANGFUSE OBSERVABILITY SUMMARY")
print("="*70)
print(f"\nSession ID: {SESSION_ID}")
print(f"\nTraces created during this session:")
print()
print("Week 1: Environment Setup")
print("  - taxonomy-definition")
print("  - data-discovery")
print("  - test-embedding-*, test-completion-*")
print()
print("Week 2: Document Processing & EDA")
print("  - process-{category} (for each category)")
print("  - eda-analysis")
print("  - entity-extraction-batch")
print()
print("Week 3: Vector Store & Embeddings")
print("  - vectorstore-init")
print("  - index-contracts_all")
print("  - embed-{doc_id} (for each document)")
print("  - semantic-search (for each query)")
print()
print("Week 4: Single Agent Design")
print("  - legal-agent-{doc_id}")
print("  - financial-agent-{doc_id}")
print("  - operational-agent-{doc_id}")
print()
print("Week 5: Multi-Agent Orchestration")
print("  - orchestration-{doc_id}")
print()
print("Week 6: Knowledge Graph")
print("  - knowledge-graph-init")
print("  - knowledge-graph-built")
print()
print(f"View all traces at: {os.environ.get('LANGFUSE_HOST')}/project")

In [ ]:
# ============================================================================
# WEEK 7.2: ADD CUSTOM SCORES
# ============================================================================

print("\nADDING CUSTOM SCORES TO TRACES")
print("="*60)

# Score the orchestration result
if 'orch_result' in globals() and orch_result:
    doc_id = orch_result['doc_id']
    overall_risk = orch_result['overall_risk']

    # Map risk to score (higher = better/lower risk)
    risk_score_map = {
        'LOW': 1.0,
        'MEDIUM': 0.7,
        'HIGH': 0.4,
        'CRITICAL': 0.1,
        'UNKNOWN': 0.5
    }
    score = risk_score_map.get(overall_risk, 0.5)

    # Create trace and add score
    score_trace = langfuse.trace(
        name=f"risk-score-{doc_id}",
        session_id=SESSION_ID,
        input={"doc_id": doc_id, "overall_risk": overall_risk}
    )

    langfuse.score(
        trace_id=score_trace.id,
        name="contract-risk-score",
        value=score,
        comment=f"Risk level: {overall_risk}"
    )

    print(f"\nScored {doc_id}:")
    print(f"  Risk Level: {overall_risk}")
    print(f"  Score: {score}")
else:
    print("No orchestration results to score.")

langfuse.flush()
print("\nScores added to Langfuse.")

## Deep Dive: Using Langfuse Scores

Scores in Langfuse enable:

| Score Type | Use Case | Example |
|------------|----------|--------|
| Quality | Evaluate output quality | 0-1 rating |
| Accuracy | Compare to ground truth | Correct/Incorrect |
| User Feedback | Thumbs up/down | 1 or 0 |
| Business Metric | Track KPIs | Risk score |

Scores can be added:
- Programmatically (as we did above)
- Via Langfuse UI annotation
- Through feedback loops from users

---
# WEEK 8: Production Deployment (YOUR WORK)
---

## Instructions

Complete the TODO cells below. Each cell has hints and instructions.
Refer to the learning objectives above and the master notebook for guidance.

---

# WEEK 8: Production Deployment

---

## Learning Objectives

By the end of Week 8, you will be able to:

- [ ] Integrate all components into a production system
- [ ] Create an interactive Gradio UI
- [ ] Test the complete end-to-end workflow
- [ ] Deploy for user interaction

---

## Key Concept: Production Systems

A production AI system needs:

```
+------------------+     +------------------+
|    User Input    | --> |   Validation     |
+------------------+     +------------------+
                              |
                              v
+------------------+     +------------------+
|   Processing     | <-- |   Retrieval      |
+------------------+     +------------------+
        |                     |
        v                     v
+------------------+     +------------------+
|   Orchestration  | --> |   Observability  |
+------------------+     +------------------+
        |
        v
+------------------+
|   Response       |
+------------------+
```

In [ ]:
# ============================================================================
# TODO 8.1: WEEK 8.1: PRODUCTION SYSTEM CLASS
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
# HINT: class ContractIntelligenceSystem:
# HINT: def __init__(self):
# HINT: def analyze_contract(self, contract_text: str, contract_id: str = "uploaded") -> Dict:
# HINT: def search_contracts(self, query: str, n_results: int = 5) -> Dict:
# HINT: def get_system_status(self) -> Dict:
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement WEEK 8.1: PRODUCTION SYSTEM CLASS
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.2: INITIALIZE PRODUCTION SYSTEM
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement INITIALIZE PRODUCTION SYSTEM
raise NotImplementedError('Complete this section')

## 8.2 Gradio Interactive UI

In [ ]:
# ============================================================================
# TODO 8.3: WEEK 8.2: GRADIO INTERFACE
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
# HINT: def analyze_contract_ui(contract_text: str) -> str:
# HINT: def search_contracts_ui(query: str) -> str:
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement WEEK 8.2: GRADIO INTERFACE
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.4: CREATE GRADIO INTERFACE
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement CREATE GRADIO INTERFACE
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.5: LAUNCH GRADIO (Conditional)
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement LAUNCH GRADIO (Conditional)
raise NotImplementedError('Complete this section')

## Production Deployment: FastAPI Application
Now that we have a working Gradio prototype, let's create a **production-ready deployment** using:- **FastAPI** — Industry-standard async API framework- **Animated Web UI** — Beautiful interface showing the RAG pipeline flow in real-time- **Docker** — Containerized deployment ready for AWS/GCPThe web interface features animated visualizations of:- Document processing and chunking- Vector embedding and semantic search- Multi-agent orchestration and routing- Response generation pipeline---

In [ ]:
# ============================================================================
# TODO 8.6: PRODUCTION DEPLOYMENT: FastAPI Application
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement PRODUCTION DEPLOYMENT: FastAPI Application
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.7: ---------------------------------------------------------------------------
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
# HINT: def _lf_trace(name: str, **kwargs):
# HINT: def _lf_generation(trace, name: str, model: str, input_data):
# HINT: def _effective_price(p: dict) -> float:
# HINT: def _product_text(p: dict) -> str:
# HINT: def _init_vector_store():
# HINT: def _batch_embed(texts: List[str]) -> List[List[float]]:
# HINT: def _embed(text: str) -> List[float]:
# HINT: def _search(query: str, n: int = 5, where: dict = None) -> List[dict]:
# HINT: def _traced_completion(messages: List[dict], trace_name: str, model: str = "gpt-4o-mini") -> str:
# HINT: def _run_price_agent(category: str, context: str) -> dict:
# HINT: def _run_catalog_agent(category: str, context: str) -> dict:
# HINT: def _run_marketing_agent(query: str, context: str) -> dict:
# HINT: def run_pipeline(query: str) -> dict:
# HINT: class QueryRequest(BaseModel):
# HINT: class QueryResponse(BaseModel):
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement ---------------------------------------------------------------------------
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.8: Week 8 Task 8
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 8
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.9: Week 8 Task 9
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 9
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.10: Week 8 Task 10
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 10
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.11: Week 8 Task 11
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 11
raise NotImplementedError('Complete this section')

---

## Production Deployment: AWS EC2 Guide

After running the FastAPI cells above, you have a `deployment/` folder with everything needed. Here's how to deploy it to AWS EC2.

### Step 1: Launch an EC2 Instance

1. Go to **AWS Console → EC2 → Launch Instance**
2. **AMI**: Ubuntu Server 22.04 LTS (Free Tier eligible)
3. **Instance type**: `t2.micro` (Free Tier) for testing, `t3.small` or `t3.medium` for production
4. **Key pair**: Create or select an existing `.pem` key pair — download it
5. **Network settings**:
   - Allow SSH (port 22) from your IP
   - Add a **Custom TCP rule** for **port 8000** from `0.0.0.0/0` (or your IP for security)
6. **Storage**: 20 GB gp3 (default is fine)
7. Click **Launch Instance**

### Step 2: Transfer Files to EC2

**Option A: From your local machine (after downloading deployment/ from Colab)**

```bash
# Make your key file secure
chmod 400 your-key.pem

# Copy the deployment folder to EC2
scp -i your-key.pem -r deployment/ ubuntu@<EC2-PUBLIC-IP>:~/app/
```

**Option B: From Google Colab**

```python
# In Colab, zip the deployment folder
!zip -r deployment.zip deployment/

# Download it to your local machine
from google.colab import files
files.download('deployment.zip')

# Then scp from local to EC2 as shown in Option A
```

### Step 3: SSH into EC2 and Set Up

```bash
# Connect to your instance
ssh -i your-key.pem ubuntu@<EC2-PUBLIC-IP>

# Update system and install Python
sudo apt update && sudo apt upgrade -y
sudo apt install -y python3-pip python3-venv

# Set up the app
cd ~/app
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```

### Step 4: Configure Environment Variables

```bash
# Create .env file with your API keys
cat > .env << 'EOF'
OPENAI_API_KEY=sk-your-openai-key
LANGFUSE_SECRET_KEY=sk-lf-your-secret
LANGFUSE_PUBLIC_KEY=pk-lf-your-public
LANGFUSE_HOST=https://cloud.langfuse.com
EOF
```

### Step 5: Launch the Application

```bash
# Test run (foreground)
uvicorn app:app --host 0.0.0.0 --port 8000

# Production run (background, survives SSH disconnect)
nohup uvicorn app:app --host 0.0.0.0 --port 8000 > app.log 2>&1 &

# Or use systemd for auto-restart:
sudo tee /etc/systemd/system/fastapi-app.service << 'EOF'
[Unit]
Description=FastAPI Competitive Intelligence App
After=network.target

[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/app
Environment="PATH=/home/ubuntu/app/venv/bin"
EnvironmentFile=/home/ubuntu/app/.env
ExecStart=/home/ubuntu/app/venv/bin/uvicorn app:app --host 0.0.0.0 --port 8000
Restart=always

[Install]
WantedBy=multi-user.target
EOF

sudo systemctl enable fastapi-app
sudo systemctl start fastapi-app
sudo systemctl status fastapi-app
```

### Step 6: Access Your Application

Open in your browser:
```
http://<EC2-PUBLIC-IP>:8000
```

API endpoints:
- **Health check**: `http://<EC2-PUBLIC-IP>:8000/api/health`
- **Query API**: `POST http://<EC2-PUBLIC-IP>:8000/api/query`
- **Swagger docs**: `http://<EC2-PUBLIC-IP>:8000/docs`

### Step 7: Verify Deployment

```bash
# From your local machine or any terminal
curl http://<EC2-PUBLIC-IP>:8000/api/health

# Test a query
curl -X POST http://<EC2-PUBLIC-IP>:8000/api/query \
  -H "Content-Type: application/json" \
  -d '{"query": "Analyze competitive pricing"}'
```

### Docker Alternative (Optional)

If you prefer Docker:

```bash
# On EC2
sudo apt install -y docker.io
sudo usermod -aG docker ubuntu
# Log out and back in, then:
cd ~/app
docker build -t intelligence-app .
docker run -d -p 8000:8000 --env-file .env --name app intelligence-app
```

### Security Recommendations for Production

- Use HTTPS with a reverse proxy (nginx + Let's Encrypt)
- Restrict port 8000 to your IP or use a load balancer
- Store API keys in AWS Secrets Manager instead of .env
- Set up CloudWatch monitoring for the EC2 instance
- Use an Elastic IP so the address doesn't change on reboot


In [ ]:
# ============================================================================
# TODO 8.12: DEPLOYMENT INSTRUCTIONS
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement DEPLOYMENT INSTRUCTIONS
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.13: FINAL CLEANUP AND SUMMARY
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement FINAL CLEANUP AND SUMMARY
raise NotImplementedError('Complete this section')

---

# Course Summary & Key Takeaways

---

## What We Built

| Week | Module | Key Technologies | Langfuse Traces |
|------|--------|------------------|------------------|
| 1 | Environment & Data | Python, Langfuse | taxonomy-definition, data-discovery |
| 2 | Document Processing | python-docx, regex | process-*, eda-analysis |
| 3 | Vector Store | ChromaDB, OpenAI | index-*, embed-*, search |
| 4 | Agent Design | Pydantic, LCEL | legal-*, financial-*, operational-* |
| 5 | Orchestration | ThreadPool, tenacity | orchestration-* |
| 6 | Knowledge Graph | NetworkX, PyVis | knowledge-graph-* |
| 7 | Observability | Langfuse scores | risk-score-* |
| 8 | Production | Gradio, integration | full-analysis-*, system-initialized |

## Architecture Summary

```
+-------------------+     +-------------------+     +-------------------+
|   Contract Input  | --> | Document Processor| --> |  Entity Extractor |
+-------------------+     +-------------------+     +-------------------+
                                  |                         |
                                  v                         v
+-------------------+     +-------------------+     +-------------------+
|   Vector Store    | <-- |    Embeddings     |     | Risk Indicators   |
+-------------------+     +-------------------+     +-------------------+
        |                                                   |
        v                                                   v
+-------------------+     +-------------------+     +-------------------+
|  Semantic Search  |     |    Orchestrator   | <-- | Agent Pool        |
+-------------------+     +-------------------+     | (Legal, Financial,|
        |                         |                 |  Operational)     |
        v                         v                 +-------------------+
+-------------------+     +-------------------+
| Knowledge Graph   |     |    Consensus      |
+-------------------+     +-------------------+
        |                         |
        +------------+------------+
                     |
                     v
            +-------------------+
            |   Gradio UI       |
            +-------------------+
                     |
                     v
            +-------------------+
            |    Langfuse       |
            | (Observability)   |
            +-------------------+
```

## Key Achievements

1. **Full Observability**: Every operation traced from start to finish
2. **Multi-Agent Architecture**: Specialized agents with parallel execution
3. **Semantic Search**: Contract retrieval beyond keyword matching
4. **Knowledge Graph**: Visual understanding of contract relationships
5. **Production Ready**: Error handling, retries, and user interface

## Skills Acquired

| Skill | Application |
|-------|-------------|
| LangChain LCEL | Building composable AI chains |
| Pydantic | Structured LLM outputs |
| ChromaDB | Vector storage and retrieval |
| Langfuse | Production observability |
| NetworkX | Knowledge graph construction |
| Gradio | Rapid UI prototyping |
| tenacity | Production-grade error handling |

## Next Steps

To extend this system:

1. **Add more agents**: Compliance, security, performance agents
2. **Enhance knowledge graph**: Add more entity types and relationships
3. **Implement feedback loop**: Score outputs and retrain prompts
4. **Add document upload**: Process user-uploaded contracts
5. **Deploy to production**: Containerize and deploy with proper auth

---

**Congratulations on completing the Contract Intelligence Capstone!**

In [ ]:
# ============================================================================
# FINAL CHECKPOINT - COMPLETE SYSTEM VERIFICATION
# ============================================================================

print("FINAL SYSTEM CHECKPOINT")
print("="*70)

final_checks = [
    # Week 1
    ("W1: Langfuse initialized", langfuse is not None),
    ("W1: Session ID created", SESSION_ID is not None),

    # Week 2
    ("W2: Document processor ready", doc_processor is not None),
    ("W2: Entity extractor ready", entity_extractor is not None),

    # Week 3
    ("W3: Vector store initialized", vector_store is not None),
    ("W3: Documents indexed", len(all_docs_flat) > 0),

    # Week 4
    ("W4: Legal agent ready", legal_agent is not None),
    ("W4: Financial agent ready", financial_agent is not None),
    ("W4: Operational agent ready", operational_agent is not None),

    # Week 5
    ("W5: Orchestrator ready", orchestrator is not None),

    # Week 6
    ("W6: Knowledge graph built", contract_kg.graph.number_of_nodes() > 0),

    # Week 8
    ("W8: Production system ready", contract_system.initialized),
    ("W8: Gradio UI defined", "demo" in globals()),
]

all_passed = True
for check_name, check_result in final_checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "="*70)
if all_passed:
    print("ALL CHECKPOINTS PASSED!")
    print("The Contract Intelligence System is fully operational.")
else:
    print("Some checkpoints failed. Review the issues above.")
print("="*70)